# Lambda × subtract_wt_baseline × n_voxels decoding sweep

Compares decoding quality across all variants submitted on 2026-03-07:

- **Model**: 31, smoothed, `--fit_responses`
- **n_voxels**: 0 (CV-R²>0), 100, 200, 500
- **subtract_wt_baseline**: off / on
- **lambda**: 0.0, 0.1, …, 1.0 (blends parametric → empirical covariance)

Primary metric: Pearson r (decoded posterior mean E vs. behavioural response x),  
averaged per subject then compared across conditions.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import pingouin as pg
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from neural_priors.utils.data import get_all_subject_ids

bids_folder = Path('/data/ds-neuralpriors')
subjects = get_all_subject_ids()
print(f'{len(subjects)} subjects')

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────

def build_key(model_label=31, smoothed=True, spherical_noise=False,
              fit_responses=False, separate_sigmas=False,
              subtract_wt_baseline=False, lambd=0.0):
    key = f'model{model_label}'
    if smoothed:             key += '.smoothed'
    if spherical_noise:      key += '.spherical_noise'
    if fit_responses:        key += '.fit_responses'
    if separate_sigmas:      key += '.separate_sigmas'
    if subtract_wt_baseline: key += '.subtract_wt_baseline'
    if lambd > 0.0:          key += f'.lambd-{lambd}'
    return key


def load_pdf(subject, key, n_voxels, roi='NPCr'):
    fn = (bids_folder / 'derivatives' / 'decoding2' / key
          / f'sub-{subject}' / 'func'
          / f'sub-{subject}_mask-{roi}_nvoxels-{n_voxels}_pars.tsv')
    pdf = pd.read_csv(fn, sep='\t', index_col=[0, 1, 2, 3, 4], header=[0, 1])
    pdf.columns.names = ['n', 'range']
    pdf.columns = pd.MultiIndex.from_arrays(
        [pdf.columns.get_level_values(0).astype(np.float32),
         pdf.columns.get_level_values(1)],
        names=pdf.columns.names)
    return pdf.sort_index(level=['session', 'run', 'trial_nr'])


def posterior_mean(pdf_slice):
    vals = pdf_slice.values
    x = pdf_slice.columns.astype(float).values
    norms = np.trapz(vals, x, axis=1)
    vals_n = vals / norms[:, None]
    E = np.trapz(vals_n * x[None, :], x, axis=1)
    mad = np.trapz(np.abs(E[:, None] - x[None, :]) * vals_n, x, axis=1)
    return E, mad


def summarize_pdf(pdf):
    E_n, mad_n = posterior_mean(pdf.xs('0.0', level='range', axis=1))
    E_w, mad_w = posterior_mean(pdf.xs('1.0', level='range', axis=1))
    range_idx = pdf.index.get_level_values('range')
    E   = np.where(range_idx == 0.0, E_n, E_w)
    mad = np.where(range_idx == 0.0, mad_n, mad_w)
    df = pdf.index.to_frame(index=False)[['session', 'run', 'trial_nr', 'x', 'range']]
    df['E']         = E
    df['mad']       = mad
    df['error']     = E - df['x']
    df['abs_error'] = np.abs(df['error'])
    df['range']     = df['range'].map({0.0: 'narrow', 1.0: 'wide'})
    return df


def subject_metrics(subject, key, n_voxels):
    try:
        pdf  = load_pdf(subject, key, n_voxels)
        pars = summarize_pdf(pdf)
    except FileNotFoundError:
        return None
    rows = []
    for rng, g in pars.groupby('range'):
        r   = pg.corr(g['E'], g['x'], method='pearson')['r'].values[0]
        mae = g['abs_error'].mean()
        rows.append({'range': rng, 'r': r, 'mae': mae})
    return pd.DataFrame(rows).assign(subject=subject)


def collect_metrics(key, n_voxels):
    results = [m for sub in subjects
               if (m := subject_metrics(sub, key, n_voxels)) is not None]
    return pd.concat(results, ignore_index=True) if results else None

## Load all variants

In [ ]:
lambda_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
n_voxels_values = [0, 100, 200, 500]

all_dfs = []

for n_voxels in tqdm(n_voxels_values, desc='n_voxels'):
    for subtract in [False, True]:
        for lam in lambda_values:
            key = build_key(fit_responses=True, subtract_wt_baseline=subtract, lambd=lam)
            df  = collect_metrics(key, n_voxels)
            if df is not None:
                df['subtract_wt_baseline'] = subtract
                df['lambda'] = lam
                df['n_voxels'] = n_voxels
                all_dfs.append(df)

data = pd.concat(all_dfs, ignore_index=True)
data['subtract_label'] = data['subtract_wt_baseline'].map(
    {False: 'raw W', True: 'baseline-subtracted W'})

print(data.groupby(['n_voxels', 'subtract_wt_baseline', 'lambda', 'range'])[['r', 'mae']]
      .mean().round(3))

## Pearson r vs lambda

In [ ]:
g = sns.FacetGrid(data, col='range', row='n_voxels', height=3.5, aspect=1.3,
                  row_order=n_voxels_values)
g.map_dataframe(sns.pointplot, x='lambda', y='r',
                hue='subtract_label', errorbar='se', dodge=False)
g.add_legend(title='WWT construction')
g.set_axis_labels('lambda (0=parametric, 1=empirical cov)', 'Pearson r')
g.set_titles('n_voxels={row_name} | {col_name}')
for ax in g.axes.flat:
    ax.axvline(x=0, color='grey', linestyle=':', linewidth=0.8)
plt.suptitle('Decoding accuracy vs lambda  (fit_responses)', y=1.01)
plt.tight_layout()
plt.savefig('lambda_r_by_subtract.png', dpi=150, bbox_inches='tight')
plt.show()

## MAE vs lambda

In [ ]:
g = sns.FacetGrid(data, col='range', row='n_voxels', height=3.5, aspect=1.3,
                  row_order=n_voxels_values)
g.map_dataframe(sns.pointplot, x='lambda', y='mae',
                hue='subtract_label', errorbar='se', dodge=False)
g.add_legend(title='WWT construction')
g.set_axis_labels('lambda', 'Mean absolute error')
g.set_titles('n_voxels={row_name} | {col_name}')
plt.suptitle('MAE vs lambda  (fit_responses)', y=1.01)
plt.tight_layout()
plt.savefig('lambda_mae_by_subtract.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap: mean r across (lambda × subtract_wt_baseline)

In [ ]:
fig, axes = plt.subplots(len(n_voxels_values), 2,
                         figsize=(13, 3.5 * len(n_voxels_values)))

for row_i, nv in enumerate(n_voxels_values):
    for col_i, rng in enumerate(['narrow', 'wide']):
        ax = axes[row_i, col_i]
        pivot = (data[(data['range'] == rng) & (data['n_voxels'] == nv)]
                 .groupby(['subtract_label', 'lambda'])['r']
                 .mean()
                 .unstack('lambda'))
        vmin = data[data['n_voxels'] == nv]['r'].quantile(0.05)
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis',
                    ax=ax, vmin=vmin)
        ax.set_title(f'n_voxels={nv} | {rng}  (mean Pearson r)')
        ax.set_xlabel('lambda')
        ax.set_ylabel('')

plt.suptitle('Mean decoding r across subjects', y=1.01)
plt.tight_layout()
plt.savefig('lambda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Best lambda per condition

In [ ]:
fig, axes = plt.subplots(len(n_voxels_values), 2,
                         figsize=(10, 3 * len(n_voxels_values)), sharey=True)

for row_i, nv in enumerate(n_voxels_values):
    for col_i, (subtract, grp) in enumerate(data[data['n_voxels'] == nv]
                                             .groupby('subtract_wt_baseline')):
        ax = axes[row_i, col_i]
        label = 'baseline-subtracted' if subtract else 'raw'
        best = (grp.groupby(['subject', 'lambda'])['r']
                   .mean().reset_index()
                   .sort_values('r', ascending=False)
                   .groupby('subject').first()['lambda'])
        best.value_counts().sort_index().plot(kind='bar', ax=ax)
        ax.set_title(f'n_voxels={nv} | {label} W')
        ax.set_xlabel('lambda')
        ax.set_ylabel('n subjects')

plt.suptitle('Best lambda per subject', y=1.01)
plt.tight_layout()
plt.show()

## Paired test: does subtract_wt_baseline help at the optimal lambda?

In [ ]:
rows = []
for nv in n_voxels_values:
    for lam in lambda_values:
        for rng in ['narrow', 'wide']:
            sub = data[(data['n_voxels'] == nv) & (data['lambda'] == lam) & (data['range'] == rng)]
            raw  = sub[~sub['subtract_wt_baseline']].set_index('subject')['r']
            sub_ = sub[sub['subtract_wt_baseline']].set_index('subject')['r']
            common = raw.index.intersection(sub_.index)
            if len(common) < 3:
                continue
            t = pg.ttest(raw[common], sub_[common], paired=True)
            rows.append({
                'n_voxels': nv, 'lambda': lam, 'range': rng,
                'r_raw': raw[common].mean(),
                'r_sub': sub_[common].mean(),
                'delta_r': sub_[common].mean() - raw[common].mean(),
                'T': t['T'].values[0],
                'p': t['p-val'].values[0],
            })

tests = pd.DataFrame(rows)
tests['sig'] = tests['p'] < 0.05
display(tests.round(4))

In [ ]:
g = sns.FacetGrid(tests, col='range', row='n_voxels', height=3.5, aspect=1.3,
                  row_order=n_voxels_values)
g.map_dataframe(sns.pointplot, x='lambda', y='delta_r', color='steelblue')
g.set_axis_labels('lambda', 'Δr  (baseline-subtracted − raw)')
g.set_titles('n_voxels={row_name} | {col_name}')
for ax in g.axes.flat:
    ax.axhline(0, color='grey', linestyle='--')
plt.suptitle('Effect of subtract_wt_baseline on decoding r', y=1.01)
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
summary = (data.groupby(['n_voxels', 'subtract_label', 'lambda', 'range'])
           .agg(n_subjects=('subject', 'nunique'),
                mean_r=('r', 'mean'), sem_r=('r', 'sem'),
                mean_mae=('mae', 'mean'), sem_mae=('mae', 'sem'))
           .round(4))
display(summary)